# 52 — Batch Scoring & Idempotency

Ejecuta scoring batch con el modelo V1 congelado. El `batch_id` depende de input + modelo + calibrador + contrato; reejecutar el mismo batch reutiliza el artefacto validado.

In [1]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.scoring import BUILD_IMPLEMENTATION, score_batch

assert SHADOW_INPUT.exists()
print("implementation:", BUILD_IMPLEMENTATION)
result = score_batch(
    input_path=SHADOW_INPUT,
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    temp_directory=TEMP_DIR,
)
result.summary

implementation: m6-52-idempotent-batch-scoring-v1


,batch_id,rows,score_min,score_max,raw_score_min,raw_score_max,review_fraction,review_rows,input_sha256,output_sha256
0,a8865bfed9374218183eb302,529054,0.0,0.130938,0.00131,0.990296,0.01,5291,dc35ed1e9c4ddee42f5d9a19255ead8fb473c007d0a1a6...,b24d1aee6d767f7cad41f8293ecfcfef8028e7b758cc7c...


In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,scoring_population_preserved,scoring,True,True,"{""input_rows"": 529054, ""scored_rows"": 529054}",batch scoring emits exactly one score row for ...
1,scoring_probabilities_valid,scoring,True,True,"{""raw_max"": 0.9902962924358858, ""raw_min"": 0.0...",raw and calibrated outputs are finite probabil...
2,scoring_output_minimizes_data,privacy,True,True,"{""forbidden_present"": []}",scoring output excludes target labels and expl...
3,ranking_contract_preserved,ranking,True,True,"{""tie_break"": [""score_desc"", ""raw_score_desc"",...",output persists deterministic rank and operati...


In [4]:
# Segunda llamada: debe resolver el mismo idempotency key y reutilizar el output.
second = score_batch(
    input_path=SHADOW_INPUT, root=paths.root, gold_spec=GOLD_SPEC, modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR, temp_directory=TEMP_DIR,
)
assert second.batch_id == result.batch_id
assert second.scored_path == result.scored_path
assert second.reused_existing
print("Idempotency replay ✅", second.batch_id)

Idempotency replay ✅ a8865bfed9374218183eb302


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("52 ✅ PASS" if result.ready_for_next_notebook else "52 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Scored batch:", result.scored_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook

52 ✅ PASS
Blocking checks: 4/4
Scored batch: /Users/macm1pro/Desktop/projects/argos/argos-code/data/production/scored_batches/a8865bfed9374218183eb302.parquet
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_scoring_a8865bfed9374218183eb302.json
